
# Mid-IR PAH features in star-forming, AGN, and composite galaxies

The 5–30 μm rest-frame spectrum showcases distinct infrared tracers:
dust polycyclic aromatic hydrocarbon (PAH) emission peaks at 6.2, 7.7, 8.6,
11.3, and 12.7 μm in star-forming galaxies, while silicate absorption
(9.7 μm Si–O stretch) and AGN heating suppress PAH and introduce continuum
growth in AGN-dominated systems. We model three templates: (a) pure starburst
(no AGN), (b) pure AGN (no star formation), and (c) composite with
AGN fraction = 0.5. the diagnostic power of mid-IR
spectroscopy: PAH strength probes star formation rate, while continuum
slope and silicate depth reveal AGN heating and dust temperature.

References:
  Smith et al. 2007, ApJ, 656, 770 (PAH feature identification).
  Hao et al. 2007, ApJL, 655, L77 (silicate absorption in AGN).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18

# =============================================================================
# Case 1: Star-forming galaxy (no AGN)
# =============================================================================
ssp = tengri.load_ssp()

model_sf = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "const",
        "all_params": tengri.FIXED,
        "log_total_mass": 10.0,  # ~32 Msun/yr over 0.3 Gyr (moderate starburst)
        "start_gyr": 0.3,
        "end_gyr": 0.0,
    },
    dust_attenuation={
        "law": "power_law",
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_diff": 0.5,  # diffuse dust: moderate optical depth
        "tau_bc": 1.2,  # birth cloud opacity: birth clouds
    },
    dust_emission={
        "type": "dale2014",  # Dale et al. 2014 dust SED
        "all_params": tengri.FIXED,
    },
    agn={
        "torus": {"type": "none"},  # No AGN torus
        "lum_ratio": 0.0,  # No AGN contribution
        "all_params": tengri.FIXED,
    },
    redshift=tengri.Fixed(0.05),
)

p_sf = dict(model_sf.spec.sample(jax.random.PRNGKey(42)))
out_sf = model_sf.predict(p_sf)
wave_sf = np.asarray(model_sf.wavelengths)
sed_sf = np.asarray(out_sf.rest_sed())
nu_l_nu_sf = C_AA_PER_S / wave_sf * sed_sf

# =============================================================================
# Case 2: Pure AGN (no star formation)
# =============================================================================
model_agn = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "const",
        "all_params": tengri.FIXED,
        "log_total_mass": 5.0,  # SFR ≈ 0 (quiescent)
        "start_gyr": 10.0,
        "end_gyr": 1.0,
    },
    dust_attenuation={
        "law": "power_law",
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_diff": 1.5,  # warm dust around AGN
        "tau_bc": 0.0,  # no birth clouds
    },
    dust_emission={
        "type": "draine_li2007",  # Draine & Li 2007 dust SED (warmer)
        "all_params": tengri.FIXED,
    },
    agn={
        "disc": {
            "type": "grahsp_sbpl",
            "all_params": tengri.FIXED,
        },
        "torus": {
            "type": "skirtor",
            "all_params": tengri.FIXED,
        },
        "log_lbol": 12.0,  # AGN bolometric luminosity: ~1e12 Lsun
        "lum_ratio": 1.0,  # 100% AGN (no stellar contribution to near/mid-IR)
        "all_params": tengri.FIXED,
    },
    redshift=tengri.Fixed(0.05),
)

p_agn = dict(model_agn.spec.sample(jax.random.PRNGKey(43)))
out_agn = model_agn.predict(p_agn)
wave_agn = np.asarray(model_agn.wavelengths)
sed_agn = np.asarray(out_agn.rest_sed())
nu_l_nu_agn = C_AA_PER_S / wave_agn * sed_agn

# =============================================================================
# Case 3: Composite (AGN fraction = 0.5)
# =============================================================================
model_composite = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "const",
        "all_params": tengri.FIXED,
        "log_total_mass": 10.2,  # ~16 Msun/yr over 1 Gyr
        "start_gyr": 1.0,
        "end_gyr": 0.0,
    },
    dust_attenuation={
        "law": "power_law",
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_diff": 1.0,
        "tau_bc": 0.8,
    },
    dust_emission={
        "type": "dale2014",
        "all_params": tengri.FIXED,
    },
    agn={
        "disc": {
            "type": "grahsp_sbpl",
            "all_params": tengri.FIXED,
        },
        "torus": {
            "type": "skirtor",
            "all_params": tengri.FIXED,
        },
        "log_lbol": 11.5,  # AGN bolometric luminosity: ~3e11 Lsun
        "lum_ratio": 0.5,  # 50% AGN, 50% stellar
        "all_params": tengri.FIXED,
    },
    redshift=tengri.Fixed(0.05),
)

p_composite = dict(model_composite.spec.sample(jax.random.PRNGKey(44)))
out_composite = model_composite.predict(p_composite)
wave_composite = np.asarray(model_composite.wavelengths)
sed_composite = np.asarray(out_composite.rest_sed())
nu_l_nu_composite = C_AA_PER_S / wave_composite * sed_composite

# =============================================================================
# Plotting: Mid-IR (5–30 μm) focus
# =============================================================================
fig, ax = plt.subplots(figsize=(7.5, 5.0))

# Rest-frame wavelength range: focus on mid-IR
wave_min, wave_max = 5e3, 3e4  # 5–30 μm in Angstroms

# Star-forming
mask_sf = (wave_sf >= wave_min) & (wave_sf <= wave_max) & (nu_l_nu_sf > 0)
ax.loglog(
    wave_sf[mask_sf],
    nu_l_nu_sf[mask_sf],
    color="C2",
    lw=2.0,
    label="Star-forming (no AGN)",
    zorder=3,
)

# Pure AGN
mask_agn = (wave_agn >= wave_min) & (wave_agn <= wave_max) & (nu_l_nu_agn > 0)
ax.loglog(
    wave_agn[mask_agn],
    nu_l_nu_agn[mask_agn],
    color="C3",
    lw=2.0,
    label="Pure AGN (no SF)",
    zorder=2,
)

# Composite
mask_composite = (
    (wave_composite >= wave_min) & (wave_composite <= wave_max) & (nu_l_nu_composite > 0)
)
ax.loglog(
    wave_composite[mask_composite],
    nu_l_nu_composite[mask_composite],
    color="C1",
    lw=2.0,
    label="Composite (AGN frac = 0.5)",
    zorder=1,
)

# Mark PAH features (Smith et al. 2007)
pah_features = [
    (6.2, "6.2 μm"),
    (7.7, "7.7 μm"),
    (8.6, "8.6 μm"),
    (11.3, "11.3 μm"),
    (12.7, "12.7 μm"),
]

for pah_um, _label in pah_features:
    pah_aa = pah_um * 1e4
    ax.axvline(pah_aa, color="0.5", lw=0.6, ls=":", alpha=0.4)

# Mark silicate absorption (9.7 μm, Hao et al. 2007)
silicate_aa = 9.7e4
ax.axvline(
    silicate_aa,
    color="0.4",
    lw=0.8,
    ls="--",
    alpha=0.5,
    label="Silicate 9.7 μm absorption",
)

# Annotations for PAH peaks (visible in SF spectrum)
ax.text(
    6.2e4,
    4e44,
    "6.2",
    fontsize=7,
    color="0.5",
    ha="center",
    alpha=0.6,
)
ax.text(
    7.7e4,
    4e44,
    "7.7",
    fontsize=7,
    color="0.5",
    ha="center",
    alpha=0.6,
)
ax.text(
    11.3e4,
    3.5e44,
    "11.3",
    fontsize=7,
    color="0.5",
    ha="center",
    alpha=0.6,
)

# Styling
ax.set(
    xlim=(5e3, 3e4),
    ylim=(1e44, 1e46),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$ [erg s$^{-1}$]",
)

ax.legend(frameon=False, fontsize=9, loc="upper left")

fig.tight_layout()
plt.savefig("plot_mid_ir_pah_features.png", dpi=150, bbox_inches="tight")